# Coastal flood step 15: inland extent of positive avoided damages

This notebook estimates **how far inland** assets with `Avoided_EAD_USD > 0` occur.

Method:
- Build one geometry per asset from coastal split geometries.
- Compute distance from each asset geometry to Jamaica coastline.
- Keep assets with positive avoided EAD.
- Summarize inland distances by scenario and sector.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

jamaica_metric_grid_crs = 'EPSG:3448'

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

scenario_damage_paths = {
    'minimum': base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates',
    'maximum': base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates',
}

out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/inland_extent_positive_avoided_assets'
out_dir.mkdir(parents=True, exist_ok=True)

print('Output folder:', out_dir)


In [ ]:
if not network_csv.exists():
    raise FileNotFoundError(f'Missing network metadata: {network_csv}')
if not shared_intersections_path.exists():
    raise FileNotFoundError(f'Missing split geometry folder: {shared_intersections_path}')
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

network_details = pd.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()
print('Network layers:', len(network_details))


In [ ]:
# Build one geometry per asset from split files.
asset_geom_rows = []
missing_split_files = []
missing_id_cols = []

for row in network_details.itertuples(index=False):
    split_file = shared_intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
    if not split_file.exists():
        missing_split_files.append(str(split_file))
        continue

    gdf = gpd.read_parquet(split_file)
    if gdf.crs is not None:
        gdf = gdf.to_crs(jamaica_metric_grid_crs)
    else:
        gdf = gdf.set_crs(jamaica_metric_grid_crs, allow_override=True)

    if row.asset_id_column not in gdf.columns:
        missing_id_cols.append((row.asset_gpkg, row.asset_layer, row.asset_id_column))
        continue

    part = gdf[[row.asset_id_column, 'geometry']].copy()
    part = gpd.GeoDataFrame(part, geometry='geometry', crs=jamaica_metric_grid_crs)
    part['Sector'] = row.sector
    part['Subsector'] = row.asset_description
    part['Asset'] = row.asset_gpkg
    part['Layer'] = row.asset_layer
    part['Asset_ID'] = part[row.asset_id_column].astype(str)

    asset_geom_rows.append(part[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'geometry']])

if not asset_geom_rows:
    raise ValueError('No asset geometries were built from split files.')

asset_geom_parts = gpd.GeoDataFrame(pd.concat(asset_geom_rows, ignore_index=True), geometry='geometry', crs=jamaica_metric_grid_crs)
asset_geoms = asset_geom_parts.dissolve(
    by=['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID'],
    as_index=False,
)
asset_geoms = gpd.GeoDataFrame(asset_geoms, geometry='geometry', crs=jamaica_metric_grid_crs)

print('Asset geometry parts:', len(asset_geom_parts))
print('Unique assets with geometry:', len(asset_geoms))
print('Missing split files:', len(missing_split_files))
print('Missing id columns in split files:', len(missing_id_cols))


In [ ]:
# Build coastline geometry (distance reference)
jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

try:
    island_geom = jamaica_boundary.geometry.union_all()
except Exception:
    island_geom = jamaica_boundary.unary_union

coastline = island_geom.boundary
print('Coastline geometry type:', coastline.geom_type)


In [ ]:
def summarize_positive_inland_extent(damage_estimates_path: Path, scenario_name: str):
    ead_file = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
    if not ead_file.exists():
        raise FileNotFoundError(f'Missing EAD file for {scenario_name}: {ead_file}')

    ead = pd.read_csv(ead_file)
    required = ['Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD']
    miss = [c for c in required if c not in ead.columns]
    if miss:
        raise KeyError(f'{scenario_name} EAD file missing columns: {miss}')

    ead = ead[required].copy()
    ead['Asset_ID'] = ead['Asset_ID'].astype(str)
    ead['Avoided_EAD_USD'] = pd.to_numeric(ead['Avoided_EAD_USD'], errors='coerce').fillna(0.0)

    merged = asset_geoms.merge(
        ead,
        on=['Asset', 'Layer', 'Asset_ID'],
        how='left',
    )
    merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

    merged['Distance_to_Coast_m'] = merged.geometry.distance(coastline)
    merged['Distance_to_Coast_km'] = merged['Distance_to_Coast_m'] / 1000.0

    positive = merged[merged['Avoided_EAD_USD'] > 0].copy()

    if positive.empty:
        overall = pd.DataFrame([{
            'Scenario': scenario_name,
            'PositiveAssetCount': 0,
            'MaxDistance_km': np.nan,
            'P95Distance_km': np.nan,
            'P99Distance_km': np.nan,
            'MedianDistance_km': np.nan,
            'MeanDistance_km': np.nan,
            'TotalAvoidedEAD_USD': 0.0,
        }])
        by_sector = pd.DataFrame(columns=['Scenario', 'Sector', 'PositiveAssetCount', 'MaxDistance_km', 'P95Distance_km', 'MedianDistance_km', 'MeanDistance_km', 'TotalAvoidedEAD_USD'])
        furthest = pd.DataFrame(columns=['Scenario', 'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'Distance_to_Coast_km'])
        return merged, positive, overall, by_sector, furthest

    overall = pd.DataFrame([{
        'Scenario': scenario_name,
        'PositiveAssetCount': int(len(positive)),
        'MaxDistance_km': float(positive['Distance_to_Coast_km'].max()),
        'P95Distance_km': float(positive['Distance_to_Coast_km'].quantile(0.95)),
        'P99Distance_km': float(positive['Distance_to_Coast_km'].quantile(0.99)),
        'MedianDistance_km': float(positive['Distance_to_Coast_km'].median()),
        'MeanDistance_km': float(positive['Distance_to_Coast_km'].mean()),
        'TotalAvoidedEAD_USD': float(positive['Avoided_EAD_USD'].sum()),
    }])

    by_sector = (
        positive.groupby('Sector', as_index=False)
        .agg(
            PositiveAssetCount=('Asset_ID', 'count'),
            MaxDistance_km=('Distance_to_Coast_km', 'max'),
            P95Distance_km=('Distance_to_Coast_km', lambda s: s.quantile(0.95)),
            MedianDistance_km=('Distance_to_Coast_km', 'median'),
            MeanDistance_km=('Distance_to_Coast_km', 'mean'),
            TotalAvoidedEAD_USD=('Avoided_EAD_USD', 'sum'),
        )
        .sort_values('MaxDistance_km', ascending=False)
        .reset_index(drop=True)
    )
    by_sector.insert(0, 'Scenario', scenario_name)

    furthest = (
        positive[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'Distance_to_Coast_km']]
        .sort_values('Distance_to_Coast_km', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
    furthest.insert(0, 'Scenario', scenario_name)

    return merged, positive, overall, by_sector, furthest


In [ ]:
results = {}
for scenario_name, damage_path in scenario_damage_paths.items():
    results[scenario_name] = summarize_positive_inland_extent(damage_path, scenario_name)

all_overall = pd.concat([results['minimum'][2], results['maximum'][2]], ignore_index=True)
all_sector = pd.concat([results['minimum'][3], results['maximum'][3]], ignore_index=True)

display(all_overall)
display(all_sector.head(20))


In [ ]:
# Save outputs
for scenario_name in ['minimum', 'maximum']:
    merged, positive, overall, by_sector, furthest = results[scenario_name]

    positive_out = out_dir / f'positive_avoided_assets_with_distance_{scenario_name}.csv'
    overall_out = out_dir / f'inland_extent_overall_summary_{scenario_name}.csv'
    sector_out = out_dir / f'inland_extent_sector_summary_{scenario_name}.csv'
    furthest_out = out_dir / f'furthest_positive_assets_{scenario_name}_top50.csv'

    positive[['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD', 'Distance_to_Coast_m', 'Distance_to_Coast_km']].to_csv(positive_out, index=False)
    overall.to_csv(overall_out, index=False)
    by_sector.to_csv(sector_out, index=False)
    furthest.to_csv(furthest_out, index=False)

    print('Saved:', positive_out)
    print('Saved:', overall_out)
    print('Saved:', sector_out)
    print('Saved:', furthest_out)

overall_compare_out = out_dir / 'inland_extent_overall_summary_minimum_vs_maximum.csv'
sector_compare_out = out_dir / 'inland_extent_sector_summary_minimum_vs_maximum.csv'

all_overall.to_csv(overall_compare_out, index=False)
all_sector.to_csv(sector_compare_out, index=False)

print('Saved:', overall_compare_out)
print('Saved:', sector_compare_out)


In [ ]:
# Chart: furthest inland distance by sector (minimum vs maximum)
sector_order = sorted(all_sector['Sector'].dropna().unique().tolist())
plot_df = all_sector.copy()
plot_df['Sector'] = pd.Categorical(plot_df['Sector'], categories=sector_order, ordered=True)
plot_df = plot_df.sort_values(['Sector', 'Scenario'])

pivot_max = plot_df.pivot(index='Sector', columns='Scenario', values='MaxDistance_km').fillna(0.0)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(pivot_max.index))
w = 0.38

ax.bar(x - w/2, pivot_max.get('minimum', pd.Series(0, index=pivot_max.index)), width=w, color='#74C476', label='Minimum')
ax.bar(x + w/2, pivot_max.get('maximum', pd.Series(0, index=pivot_max.index)), width=w, color='#238B45', label='Maximum')

ax.set_xticks(x)
ax.set_xticklabels([str(s).capitalize() for s in pivot_max.index], rotation=25, ha='right')
ax.set_ylabel('Furthest inland distance (km)')
ax.set_title('Furthest inland positive avoided-damage assets by sector')
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.legend(frameon=False)

for i, v in enumerate(pivot_max.get('minimum', pd.Series(0, index=pivot_max.index))):
    ax.text(i - w/2, v + 0.05, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
for i, v in enumerate(pivot_max.get('maximum', pd.Series(0, index=pivot_max.index))):
    ax.text(i + w/2, v + 0.05, f'{v:.1f}', ha='center', va='bottom', fontsize=8)

chart_out = out_dir / 'furthest_inland_distance_by_sector_minimum_vs_maximum.png'
fig.savefig(chart_out, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', chart_out)


In [ ]:
# Quick headline numbers
for scenario_name in ['minimum', 'maximum']:
    overall = results[scenario_name][2].iloc[0]
    print(f"{scenario_name.capitalize()}: max inland distance = {overall['MaxDistance_km']:.2f} km, p95 = {overall['P95Distance_km']:.2f} km, positive assets = {int(overall['PositiveAssetCount'])}")
